# Intent Classification Chatbot — CLINC150
### TF-IDF + Naive Bayes + Logistic Regression

In [10]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

install('datasets')
install('scikit-learn')
install('nltk')
install('seaborn')
install('matplotlib')



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os, re, csv, pickle, random, warnings, time
from datetime import datetime
from collections import Counter

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from datasets import load_dataset

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt',      quiet=True)
nltk.download('stopwords',  quiet=True)
nltk.download('wordnet',    quiet=True)
nltk.download('punkt_tab',  quiet=True)
nltk.download('omw-1.4',    quiet=True)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')


  error: subprocess-exited-with-error
  
  × Building wheel for backports.lzma (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [40 lines of output]
      This is backports.lzma version 0.0.14
      /private/var/folders/yb/1vxfk8_x2fbdnqhyn4mynsnr0000gn/T/pip-build-env-oqe4lrmz/overlay/lib/python3.13/site-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: BSD License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      running bdist_wheel
     

CalledProcessError: Command '['/Users/laleshkumar/Python_course/.venv/bin/python', '-m', 'pip', 'install', 'backports.lzma', '-q']' returned non-zero exit status 1.

## Load Dataset

In [13]:
dataset = load_dataset('DeepPavlov/clinc_oos', 'plus')

train_df = pd.DataFrame(dataset['train'])
valid_df = pd.DataFrame(dataset['validation'])
test_df  = pd.DataFrame(dataset['test'])

intent_names = dataset['train'].features['intent'].names

train_df['intent_name'] = train_df['intent'].map(lambda x: intent_names[x])
valid_df['intent_name'] = valid_df['intent'].map(lambda x: intent_names[x])
test_df['intent_name']  = test_df['intent'].map(lambda x: intent_names[x])

train_df.head()


NameError: name 'load_dataset' is not defined

In [ ]:
print(f'Train : {len(train_df):,}')
print(f'Valid : {len(valid_df):,}')
print(f'Test  : {len(test_df):,}')
print(f'Intents: {len(intent_names)}')


## EDA

In [ ]:
train_df['text_length'] = train_df['text'].apply(len)
train_df['word_count']  = train_df['text'].apply(lambda x: len(x.split()))

train_df[['text_length', 'word_count']].describe().round(2)


In [ ]:
intent_counts = train_df['intent_name'].value_counts()
intent_counts.head(10)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

top30 = intent_counts.head(30)
axes[0].barh(top30.index[::-1], top30.values[::-1], color=sns.color_palette('husl', 30))
axes[0].set_xlabel('Training Samples')
axes[0].set_title('Top 30 Intents by Frequency')
axes[0].axvline(x=top30.mean(), color='red', linestyle='--', label=f'Mean = {top30.mean():.0f}')
axes[0].legend()

axes[1].hist(train_df['text_length'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(train_df['text_length'].mean(),   color='red',   linestyle='--', label=f'Mean = {train_df["text_length"].mean():.1f}')
axes[1].axvline(train_df['text_length'].median(), color='green', linestyle='--', label=f'Median = {train_df["text_length"].median():.1f}')
axes[1].set_xlabel('Text Length (characters)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Query Text Lengths')
axes[1].legend()

plt.tight_layout()
plt.show()


## Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def preprocess_text(text):
    text   = text.lower()
    text   = re.sub(r'http\S+|www\.\S+', '', text)
    text   = re.sub(r'[^a-z0-9\s]', '', text)
    text   = re.sub(r'\b\d+\b', '', text)
    text   = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return ' '.join(tokens)


In [ ]:
train_df['clean_text'] = train_df['text'].apply(preprocess_text)
valid_df['clean_text'] = valid_df['text'].apply(preprocess_text)
test_df['clean_text']  = test_df['text'].apply(preprocess_text)

train_df[['text', 'clean_text', 'intent_name']].sample(5, random_state=7)


## TF-IDF Vectorization

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['clean_text'])
X_valid_tfidf = tfidf_vectorizer.transform(valid_df['clean_text'])
X_test_tfidf  = tfidf_vectorizer.transform(test_df['clean_text'])

print(f'Train shape : {X_train_tfidf.shape}')
print(f'Valid shape : {X_valid_tfidf.shape}')
print(f'Test shape  : {X_test_tfidf.shape}')


## Label Encoding

In [ ]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df['intent_name'])
y_valid = label_encoder.transform(valid_df['intent_name'])
y_test  = label_encoder.transform(test_df['intent_name'])

num_classes = len(label_encoder.classes_)
print(f'Total classes: {num_classes}')


## Train Naive Bayes

In [ ]:
nb_model = MultinomialNB(alpha=0.1)

start = time.time()
nb_model.fit(X_train_tfidf, y_train)
train_time_nb = time.time() - start

test_pred_nb = nb_model.predict(X_test_tfidf)
print(f'NB Train Time : {train_time_nb:.2f}s')
print(f'NB Test Acc   : {accuracy_score(y_test, test_pred_nb)*100:.2f}%')


## Train Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    C=5.0,
    solver='saga',
    multi_class='multinomial',
    n_jobs=-1,
    random_state=42
)

start = time.time()
lr_model.fit(X_train_tfidf, y_train)
train_time_lr = time.time() - start

test_pred_lr = lr_model.predict(X_test_tfidf)
print(f'LR Train Time : {train_time_lr:.2f}s')
print(f'LR Test Acc   : {accuracy_score(y_test, test_pred_lr)*100:.2f}%')


## Model Evaluation

In [ ]:
def evaluate_model(name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred,    average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred,        average='weighted', zero_division=0)
    print(f'{name}')
    print(f'  Accuracy  : {acc*100:.2f}%')
    print(f'  Precision : {prec*100:.2f}%')
    print(f'  Recall    : {rec*100:.2f}%')
    print(f'  F1 Score  : {f1*100:.2f}%')
    return {'model': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

nb_results = evaluate_model('Naive Bayes',         y_test, test_pred_nb)
print()
lr_results = evaluate_model('Logistic Regression', y_test, test_pred_lr)


In [ ]:
comparison_df = pd.DataFrame([
    {'Model': nb_results['model'], 'Accuracy': f"{nb_results['accuracy']*100:.2f}%",
     'F1': f"{nb_results['f1']*100:.2f}%", 'Train Time': f'{train_time_nb:.2f}s'},
    {'Model': lr_results['model'], 'Accuracy': f"{lr_results['accuracy']*100:.2f}%",
     'F1': f"{lr_results['f1']*100:.2f}%", 'Train Time': f'{train_time_lr:.2f}s'},
])
comparison_df


In [ ]:
# Pick best model
if lr_results['f1'] >= nb_results['f1']:
    best_model, best_model_name = lr_model, 'Logistic Regression'
else:
    best_model, best_model_name = nb_model, 'Naive Bayes'

print(f'Best model: {best_model_name}')


In [ ]:
# Confusion matrix — top 15 intents
top15_intents = pd.Series(label_encoder.inverse_transform(y_test)).value_counts().head(15).index.tolist()
top15_encoded = label_encoder.transform(top15_intents)

mask         = np.isin(y_test, top15_encoded)
y_test_sub   = y_test[mask]
y_pred_sub   = test_pred_lr[mask]

cm = confusion_matrix(y_test_sub, y_pred_sub, labels=top15_encoded)

plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=[n[:20] for n in top15_intents],
            yticklabels=[n[:20] for n in top15_intents],
            cmap='Blues', linewidths=0.5)
plt.title('Confusion Matrix — Logistic Regression (Top 15 Intents)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()


## Save Model

In [ ]:
with open('best_model.pkl',       'wb') as f: pickle.dump(best_model,       f)
with open('tfidf_vectorizer.pkl', 'wb') as f: pickle.dump(tfidf_vectorizer, f)
with open('label_encoder.pkl',    'wb') as f: pickle.dump(label_encoder,    f)

print('Saved: best_model.pkl, tfidf_vectorizer.pkl, label_encoder.pkl')


## Response System

In [ ]:
INTENT_RESPONSES = {
    'transfer'     : ["I can help you transfer money. Who is the recipient and how much?",
                       "Sure! Provide the account number and amount to transfer.",
                       "Transfer request received. Please verify recipient details."],
    'balance'      : ["Your current balance is $2,450.75.",
                       "Available balance: $2,450.75 as of today.",
                       "Balance check complete: $2,450.75."],
    'transactions' : ["Here are your last 5 transactions: [details].",
                       "What date range would you like for your transaction history?",
                       "Fetching recent transactions..."],
    'freeze_account':["Your account has been frozen. Contact support to unfreeze.",
                       "Account freeze initiated — locked for your safety."],
    'pin_change'   : ["Go to Settings > Security > Change PIN.",
                       "Verify your identity first, then I'll guide you through PIN change."],
    'card_declined': ["Card declined — check balance, expiry, or security flag.",
                       "Common reasons: insufficient funds, expired card, or bank security block."],
    'credit_limit' : ["Your credit limit is $5,000. Want to request an increase?",
                       "Credit limit: $5,000 total, $3,200 available."],
    'rewards'      : ["You have 2,450 reward points worth ~$24.50.",
                       "Reward points: 2,450. Redeem for cashback or travel miles!"],
    'pay_bill'     : ["Which biller and how much? I'll set it up.",
                       "Bill payment ready. Confirm the amount and biller name."],
    'loan'         : ["Personal loans from $1,000–$50,000 at competitive rates.",
                       "Loan info available. What type are you interested in?"],
    'interest_rate': ["Savings: 3.5% APY | Loans from 6.9% APR | Credit Card: 18.9%.",
                       "Visit our rates page for the most current information."],
    'exchange_rate': ["USD/EUR 0.92 | USD/GBP 0.79 | USD/INR 83.5 (today).",
                       "Which currencies do you need? I'll fetch the latest rates."],
    'currency'     : ["Which currencies and amount? I'll convert it for you.",
                       "Currency conversion ready. Specify the amount and currencies."],
    'greeting'     : ["Hello! How can I assist you today?",
                       "Hi there! What can I help you with?",
                       "Hey! Ready to help with any queries."],
    'goodbye'      : ["Goodbye! Have a wonderful day!", "Bye! Feel free to return anytime."],
    'thank_you'    : ["You're welcome! Anything else?", "Happy to help!"],
    'what_can_i_ask_you': ["Ask me about balances, transfers, bills, loans, travel, and more!",
                            "I handle 150+ query types — banking, travel, food, fun, and more!"],
    'time'         : [f"Current time: {datetime.now().strftime('%H:%M')}.",
                       f"It's {datetime.now().strftime('%I:%M %p')}."],
    'date'         : [f"Today is {datetime.now().strftime('%B %d, %Y')}.",
                       f"{datetime.now().strftime('%A, %d %B %Y')}."],
    'alarm'        : ["What time should I set the alarm for?"],
    'weather'      : ["Check weather.com or your phone's weather app for the latest forecast."],
    'calculator'   : ["Give me the numbers and I'll calculate."],
    'definition'   : ["Which word would you like defined?"],
    'flight_status': ["Provide your flight number and I'll check the status."],
    'book_flight'  : ["Where and when are you traveling? Let's book your flight!"],
    'restaurant_suggestion': ["What cuisine and your location? I'll suggest nearby options."],
    'recipe'       : ["Which dish? I'll find a recipe for you."],
    'tell_joke'    : ["Why don't scientists trust atoms? They make up everything! 😄",
                       "What do you call a fake noodle? An impasta! 😂",
                       "What's a computer's favorite snack? Microchips! 💻"],
    'who_made_you' : ["I'm an AI chatbot built with Python, TF-IDF, and scikit-learn!",
                       "A student built me as a B.Tech NLP project using CLINC150."],
    'meaning_of_life': ["The answer is 42! (Douglas Adams 😄)", "Life's meaning is what you make of it!"],
    'oos'          : ["I didn't understand that. Could you rephrase?",
                       "That's outside my expertise. Try asking differently!"],
}

DEFAULT_RESPONSES = [
    "I understand your query about {intent}. Let me help!",
    "Regarding {intent}: I'll do my best to assist.",
]

def get_response(intent_name):
    for key in INTENT_RESPONSES:
        if key in intent_name.lower() or intent_name.lower() in key:
            return random.choice(INTENT_RESPONSES[key])
    return random.choice(DEFAULT_RESPONSES).format(intent=intent_name.replace('_', ' '))

print('Response system ready.')


## Chatbot Pipeline

In [ ]:
class IntentChatbot:
    def __init__(self, model, vectorizer, encoder, threshold=0.35):
        self.model    = model
        self.vectorizer = vectorizer
        self.encoder  = encoder
        self.threshold = threshold
        self.history  = []

    def predict_intent(self, text):
        cleaned   = preprocess_text(text)
        features  = self.vectorizer.transform([cleaned])
        pred      = self.model.predict(features)[0]
        conf      = self.model.predict_proba(features)[0].max()
        intent    = self.encoder.inverse_transform([pred])[0]
        return intent, conf

    def respond(self, user_input):
        if not user_input.strip():
            return 'Please type something!', 'unknown', 0.0
        intent, conf = self.predict_intent(user_input)
        if conf < self.threshold:
            response = '⚠️ Sorry, I did not understand. Please rephrase.'
            intent   = 'unknown'
        else:
            response = get_response(intent)
        self.history.append({
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'user_query': user_input, 'intent': intent,
            'confidence': round(conf * 100, 2), 'response': response
        })
        return response, intent, conf


chatbot = IntentChatbot(best_model, tfidf_vectorizer, label_encoder, threshold=0.35)
print(f'Chatbot ready  |  Model: {best_model_name}  |  Intents: {num_classes}')


In [ ]:
query = 'What is my account balance?'
response, intent, conf = chatbot.respond(query)
print(f'Query      : {query}')
print(f'Intent     : {intent}')
print(f'Confidence : {conf*100:.1f}%')
print(f'Response   : {response}')


## Fallback Handling

In [ ]:
test_queries = [
    'What is my account balance?',
    'Transfer money to Rahul',
    'My card was declined',
    'asdfghj random gibberish xyz',
    'quantum physics black holes',
]

for q in test_queries:
    r, i, c = chatbot.respond(q)
    tag = '✅' if i != 'unknown' else '⚠️'
    print(f'{tag} [{c*100:5.1f}%] "{q[:45]}"')
    print(f'         Intent: {i}  |  Response: {r[:60]}')
    print()


## Live Chat

In [ ]:
EXIT_COMMANDS = {'exit', 'quit', 'bye', 'goodbye', 'stop'}

print('=' * 55)
print(f'  CHATBOT LIVE  |  Model: {best_model_name}')
print('  Type exit / quit / bye to stop')
print('=' * 55)

while True:
    try:
        user_input = input('You: ').strip()
        if not user_input:
            continue
        if user_input.lower() in EXIT_COMMANDS:
            print('Bot: Goodbye! 👋')
            break
        response, intent, conf = chatbot.respond(user_input)
        print(f'Bot [{conf*100:.0f}%]: {response}\n')
    except (KeyboardInterrupt, EOFError):
        print('\nBot: Chat ended. Goodbye!')
        break


## Save Conversation Log

In [ ]:
def save_history(chatbot, filename='conversation_history.csv'):
    if not chatbot.history:
        print('No history to save.')
        return
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['timestamp','user_query','intent','confidence','response'])
        writer.writeheader()
        writer.writerows(chatbot.history)
    print(f'Saved {len(chatbot.history)} messages → {filename}')

save_history(chatbot)
pd.read_csv('conversation_history.csv')
